In [36]:
import os
import json
import base64
import requests
import pandas as pd
import time
import re 
from pathlib import Path
from PIL import Image
from io import BytesIO
from openai import OpenAI

# --- Configuration ---
OPENROUTER_API_KEY = "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"
MODEL_NAME = "google/gemma-3-27b-it:free" 
MAX_IMAGE_SIZE = 800

NGROK_URL = "https://3325-194-27-149-203.ngrok-free.app/v1"
# MODELS_TO_TEST = ["llava:7b", "qwen2.5vl:7b", "qwen3-vl:8b"]
# MODELS_TO_TEST = ["llava:7b"]
# MODELS_TO_TEST = ["qwen2.5vl:7b"]
MODELS_TO_TEST = ["qwen3-vl:8b"]

In [42]:
#  --- 1. Image Encoding ---
def encode_and_resize_image(image_path):
    try:
        with Image.open(image_path) as img:
            if img.mode != 'RGB':
                img = img.convert('RGB')
            buffer = BytesIO()
            img.save(buffer, format="JPEG", quality=85)
            base64_str = base64.b64encode(buffer.getvalue()).decode('utf-8')
            return f"data:image/jpeg;base64,{base64_str}"
    except Exception as e:
        print(f"❌ Error processing image {image_path}: {e}")
        return None

# --- Configuration for Ngrok ---
dataset_path = Path("/Users/smeh/Desktop/Thesis/DChartQA/ChartQA Dataset")

client = OpenAI(
    base_url=NGROK_URL,
    api_key="ollama",
    timeout=180.0 
)

# --- 2. API Queries ---
def query_vision_model(model_name, prompt, base64_image, max_retries=10):
    """Sends the image and prompt to the VLM via ngrok, returning answer and time."""
    
    # Strict instructions for Chart VQA
    strict_prompt = f"""{prompt}

You are an expert data analyst. Answer the user's question using only the provided chart.

CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:
1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler (e.g., never write "The answer is" or "Based on the chart").
2. For numerical answers, output just the exact number and unit if applicable (e.g., "45%" or "$120").
3. For Yes/No questions, output strictly "Yes" or "No".
4. For multiple items, separate them only with commas (e.g., "France, Germany" not "France and Germany").

"""
# Few shot approach
# EXAMPLES:
# Question: What is the highest value?
# Answer: 75%
# Question: Is the blue bar taller than the red bar?
# Answer: Yes
# Question: Which two months had the lowest sales?
# Answer: Jan, Feb
    
    start_time = time.time()
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": strict_prompt},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                        ]
                    }
                ]
            )
            result_text = response.choices[0].message.content.strip()
            duration = time.time() - start_time
            return result_text, duration
            
        except Exception as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    return "ERROR_MAX_RETRIES_REACHED", (time.time() - start_time)

# --- 3. Dataset Loading ---
def load_chart_qa_split(dataset_path, split_name, type="human"):
    """Loads ChartQA data for a specific split (train, val, test)."""
    json_file = os.path.join(dataset_path, split_name, f"{split_name}_{type}.json")
    image_dir = os.path.join(dataset_path, split_name, "png")
    
    if not os.path.exists(json_file):
        raise FileNotFoundError(f"Cannot find dataset at: {json_file}")

    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    df = pd.DataFrame(data)
    df['image_path'] = df['imgname'].apply(lambda x: os.path.join(image_dir, x))
    return df

# --- 4. Processing Pipeline & Evaluation ---
def evaluate_chartqa(ans, gt):
    """Improved evaluation for numbers (incl percentages/decimals) and lists."""
    import re
    
    ans_clean = ans.lower().strip()
    gt_clean = gt.lower().strip()
    
    # 1. Direct match
    if ans_clean == gt_clean: return True
    
    # 2. Advanced Numerical comparison (supports % vs decimal)
    def extract_numerical_value(s):
        # Find something that looks like a number, possibly followed by %
        match = re.search(r'(-?\d+\.?\d*)\s*(%?)', s.replace(",", ""))
        if match:
            val = float(match.group(1))
            if match.group(2) == '%':
                val /= 100.0
            return val
        return None

    ans_val = extract_numerical_value(ans_clean)
    gt_val = extract_numerical_value(gt_clean)
    
    if ans_val is not None and gt_val is not None:
        # Use a small epsilon for float comparison
        if abs(ans_val - gt_val) < 1e-5:
            return True

    # 3. Set-based comparison for lists (ignoring order and separating chars)
    # Extracts all alphanumeric words - handles "Belgium Switzerland" vs "[Switzerland, Belgium]"
    ans_set = set(re.findall(r'\w+', ans_clean))
    gt_set = set(re.findall(r'\w+', gt_clean))
    
    if ans_set == gt_set and len(gt_set) > 0:
        return True
        
    # 4. Fallback substring
    return gt_clean.replace(" ", "") in ans_clean.replace(" ", "")

def process_chartqa_to_csv(df):
    for model_name in MODELS_TO_TEST:
        print(f"\n🚀 Starting inference on {len(df)} questions using {model_name}...")
        
        safe_model_name = model_name.replace("/", "_").replace(":", "_")
        output_csv = f"chartqa_{safe_model_name}_results.csv"
        
        processed_tasks = set()
        file_exists = os.path.isfile(output_csv)
        
        if file_exists:
            try:
                existing_df = pd.read_csv(output_csv)
                if not existing_df.empty and 'task_id' in existing_df.columns:
                    processed_tasks = set(existing_df['task_id'].astype(str) + "_" + existing_df['prompt'].astype(str))
                    print(f"🔄 Resuming... Skipping {len(processed_tasks)} tasks.")
            except Exception as e:
                print(f"⚠️ Resume error: {e}")
                
        for index, row in df.iterrows():
            task_id, query, ground_truth = str(row['imgname']), str(row['query']), str(row['label']).strip()
            task_key = f"{task_id}_{query}"
            
            if task_key in processed_tasks: continue 
                
            print(f"\nProcessing: {task_id} | {query}")
            
            base64_img = encode_and_resize_image(row['image_path'])
            if not base64_img: continue
                
            # Perform inference and capture time
            model_answer, duration = query_vision_model(model_name, query, base64_img.split(",", 1)[1])
            
            # Evaluate using refined logic
            evaluation_passed = evaluate_chartqa(model_answer, ground_truth)
            
            # Save to CSV (now including run_time)
            result_df = pd.DataFrame([{
                'task_id': task_id,
                'prompt': query,
                'model_answer': model_answer,
                'ground_truth': ground_truth,
                'evaluation': evaluation_passed,
                'run_time': round(duration, 3) 
            }])
            
            result_df.to_csv(output_csv, mode='a', header=not file_exists, index=False)
            file_exists = True 
            processed_tasks.add(task_key)
            
            eval_mark = "✅" if evaluation_passed else "❌"
            print(f"   {eval_mark} Evaluation | Time: {duration:.2f}s | Reply: '{model_answer}'")
            time.sleep(5)

# --- Execution ---
if __name__ == "__main__":
    current_dir = os.getcwd()
    dynamic_dataset_path = os.path.abspath(os.path.join(current_dir, "..", "DChartQA", "ChartQA Dataset"))

    try:
        train_df = load_chart_qa_split(dynamic_dataset_path, "train", type="human")
        if train_df is not None and not train_df.empty:
            print(f"✅ Loaded {len(train_df)} questions.")
            process_chartqa_to_csv(train_df.head(100))
        else:
            print("⚠️ Dataset empty.")
    except Exception as e:
        print(f"❌ Fatal Error: {e}")


✅ Loaded 7398 questions.

🚀 Starting inference on 100 questions using qwen3-vl:8b...
🔄 Resuming... Skipping 59 tasks.

Processing: 34.png | What is the second least difference in the two voters' opinions?
   ✅ Evaluation | Time: 40.81s | Reply: '29'

Processing: 3886.png | What's the value of blue graph in 2014?
   ✅ Evaluation | Time: 3.69s | Reply: '19%'

Processing: 3886.png | In which year the values of blue and green graph are same?


KeyboardInterrupt: 

In [ ]:
#  --- 1. Image Encoding ---
def encode_and_resize_image(image_path, max_size=MAX_IMAGE_SIZE):
    try:
        with Image.open(image_path) as img:
            if img.mode != 'RGB':
                img = img.convert('RGB')
            img.thumbnail((max_size, max_size))
            buffer = BytesIO()
            img.save(buffer, format="JPEG", quality=85)
            base64_str = base64.b64encode(buffer.getvalue()).decode('utf-8')
            return f"data:image/jpeg;base64,{base64_str}"
    except Exception as e:
        print(f"❌ Error processing image {image_path}: {e}")
        return None

# --- 2. API Queries ---
def query_vision_model(prompt, base64_image, max_retries=3):
    """Sends the image and prompt to the VLM, forcing a short answer."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    
    # Strict instructions for Chart VQA
    strict_prompt = f"""{prompt}

You are an expert data analyst. Answer the user's question using only the provided chart.

CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:
1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler (e.g., never write "The answer is" or "Based on the chart").
2. For numerical answers, output just the exact number and unit if applicable (e.g., "45%" or "$120").
3. For Yes/No questions, output strictly "Yes" or "No".
4. For multiple items, separate them only with commas (e.g., "France, Germany" not "France and Germany").
"""
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": strict_prompt},
                    {"type": "image_url", "image_url": {"url": base64_image}}
                ]
            }
        ]
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.post(
                url="https://openrouter.ai/api/v1/chat/completions",
                headers=headers,
                json=payload,
                timeout=120 
            )
            response.raise_for_status() 
            result_text = response.json()['choices'][0]['message']['content'].strip()
            return result_text
            
        except requests.exceptions.Timeout:
            print(f"   ⏳ Attempt {attempt + 1}/{max_retries}: Request timed out.")
        except requests.exceptions.RequestException as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    return "ERROR_MAX_RETRIES_REACHED"

# --- 3. Dataset Loading ---
def load_chart_qa_split(dataset_path, split_name, type="human"):
    """
    Loads ChartQA data for a specific split (train, val, test).
    """
    json_file = os.path.join(dataset_path, split_name, f"{split_name}_{type}.json")
    image_dir = os.path.join(dataset_path, split_name, "png")
    
    if not os.path.exists(json_file):
        raise FileNotFoundError(f"Cannot find dataset at: {json_file}\nPlease check your folder structure.")

    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    df = pd.DataFrame(data)
    df['image_path'] = df['imgname'].apply(lambda x: os.path.join(image_dir, x))
    return df

# --- 4. Processing Pipeline & Evaluation ---
def process_chartqa_to_csv(df):
    print(f"🚀 Starting inference on {len(df)} questions using {MODEL_NAME}...")
    
    safe_model_name = MODEL_NAME.replace("/", "_").replace(":", "_")
    output_csv = f"chartqa_{safe_model_name}_results.csv"
    
    processed_tasks = set()
    file_exists = os.path.isfile(output_csv)
    
    # Resume capabilities
    if file_exists:
        try:
            existing_df = pd.read_csv(output_csv)
            if not existing_df.empty and 'task_id' in existing_df.columns and 'prompt' in existing_df.columns:
                processed_tasks = set(existing_df['task_id'].astype(str) + "_" + existing_df['prompt'].astype(str))
                print(f"🔄 Found existing CSV! Resuming... Skipping {len(processed_tasks)} already processed tasks.")
        except Exception as e:
            print(f"⚠️ Could not read existing CSV for progress tracking: {e}")
            
    for index, row in df.iterrows():
        task_id = str(row['imgname'])
        query = str(row['query'])
        ground_truth = str(row['label']).strip()
        
        task_key = task_id + "_" + query
        if task_key in processed_tasks:
            continue 
            
        print(f"\nProcessing Task: {task_id} | Question: {query}")
        
        # 1. Vision Request
        base64_img = encode_and_resize_image(row['image_path'])
        if not base64_img:
            print("   ⚠️ Skipping due to image error.")
            continue
            
        model_answer = query_vision_model(query, base64_img)
        
        # --- NEW EVALUATION LOGIC ---
        # 1. Remove spaces and make lowercase
        gt_clean = ground_truth.lower().replace(" ", "")
        ans_clean = model_answer.lower().replace(" ", "")
        
        evaluation_passed = False
        
        # 2. Try to evaluate as a numerical match first
        try:
            gt_value = float(gt_clean) 
            
            # Extract all numbers (including decimals and negatives) from the model's answer
            numbers_in_ans = re.findall(r'-?\d+\.?\d*', ans_clean)
            
            # Check if any number the model output mathematically matches the ground truth
            for num_str in numbers_in_ans:
                if float(num_str) == gt_value:
                    evaluation_passed = True
                    break
        except ValueError:
            pass # The ground truth is text, not a number. Move on to step 3.
            
        # 3. Text fallback
        if not evaluation_passed:
            evaluation_passed = gt_clean in ans_clean
        # ----------------------------
        
        # 3. Save to CSV
        result_df = pd.DataFrame([{
            'task_id': task_id,
            'prompt': query,
            'model_answer': model_answer,
            'ground_truth': ground_truth,
            'evaluation': evaluation_passed
        }])
        
        result_df.to_csv(output_csv, mode='a', header=not file_exists, index=False)
        file_exists = True 
        
        processed_tasks.add(task_key)
        eval_mark = "✅" if evaluation_passed else "❌"
        print(f"   {eval_mark} Evaluation: {'Passed' if evaluation_passed else 'Failed'} (Truth: '{ground_truth}' | Model: '{model_answer}')")
        
        # Pause to respect API rate limits
        time.sleep(5)

# --- Execution ---
if __name__ == "__main__":
    current_dir = os.getcwd()
    dynamic_dataset_path = os.path.abspath(os.path.join(current_dir, "..", "DChartQA", "ChartQA Dataset"))

    try:
        print("Loading ChartQA dataset...")
        train_df = load_chart_qa_split(dynamic_dataset_path, "train", type="human")
        
        if train_df is not None and not train_df.empty:
            print(f"✅ Successfully loaded {len(train_df)} questions.\n")
            print("=" * 60)
            
            process_chartqa_to_csv(train_df.sample(3))
            
            print("\n🎉 Run complete!")
        else:
            print("⚠️ The dataset loaded, but it appears to be empty.")
            
    except FileNotFoundError as e:
        print(e)

Loading ChartQA dataset...
✅ Successfully loaded 7398 questions.

🚀 Starting inference on 3 questions using google/gemma-3-27b-it:free...

Processing Task: 6149.png | Question: What color represents Democrats?
   ❌ Attempt 1/3: API Request failed: 429 Client Error: Too Many Requests for url: https://openrouter.ai/api/v1/chat/completions
   ❌ Attempt 2/3: API Request failed: 429 Client Error: Too Many Requests for url: https://openrouter.ai/api/v1/chat/completions
   ❌ Attempt 3/3: API Request failed: 429 Client Error: Too Many Requests for url: https://openrouter.ai/api/v1/chat/completions
   ❌ Evaluation: Failed (Truth: 'blue' | Model: 'ERROR_MAX_RETRIES_REACHED')

Processing Task: 10832.png | Question: What is the difference between maximum and minimum value of yellow bar?
   ❌ Attempt 1/3: API Request failed: 429 Client Error: Too Many Requests for url: https://openrouter.ai/api/v1/chat/completions


KeyboardInterrupt: 